# 1. Microstructure Research

流程: Orderbook 散点图 -> Wall Mid -> Normalized 图 -> Return ACF -> 交易分类 -> Adverse Selection

In [57]:
import sys
sys.path.insert(0, '.')

from utils.dataio import load_prices, load_prices_wide, load_trades
from utils.fair import build_outer_inner_wall_mid
from utils.orderbook import compute_spread, return_autocorrelation, adverse_selection, order_interval_distribution, trade_interval_distribution, normalized_level_trade_stats
from utils.viz import plot_orderbook_scatter, plot_autocorrelation, plot_trade_profile, plot_normalized_orderbook, plot_interval_distribution
import polars as pl
import numpy as np

In [ ]:
# === 配置 ===
ROUND = 2
DAY = 0
PRODUCT = 'ASH_COATED_OSMIUM'  # 改成你要分析的产品

# 逆向工程过滤开关（按量过滤）
FILTER_BY_VOLUME = True
MIN_ORDER_VOLUME = 10

MIN_TRADE_QUANTITY = 9
MAX_TRADE_QUANTITY = 10  # 例如 30；None 表示不设上限

## Step 1: 加载数据

In [59]:
prices_long = load_prices(ROUND, DAY)
prices_wide = load_prices_wide(ROUND, DAY)
trades = load_trades(ROUND, DAY)

# 对成交做上下限过滤：abs(qty) in [MIN_TRADE_QUANTITY, MAX_TRADE_QUANTITY]
if FILTER_BY_VOLUME:
    qty_col = 'quantity' if 'quantity' in trades.columns else ('volume' if 'volume' in trades.columns else None)
    if qty_col is not None:
        before_n = trades.height
        if MIN_TRADE_QUANTITY is not None:
            trades = trades.filter(pl.col(qty_col).abs() >= MIN_TRADE_QUANTITY)
        if MAX_TRADE_QUANTITY is not None:
            trades = trades.filter(pl.col(qty_col).abs() <= MAX_TRADE_QUANTITY)
        print(f"Trades filtered by {qty_col}: {before_n} -> {trades.height} (min={MIN_TRADE_QUANTITY}, max={MAX_TRADE_QUANTITY})")

print('Products:', prices_long['product'].unique().to_list())
print('Prices shape:', prices_long.shape)
print('Trades shape:', trades.shape)
trades.head()

Trades filtered by quantity: 803 -> 23 (min=10, max=10)
Products: ['INTARIAN_PEPPER_ROOT', 'ASH_COATED_OSMIUM']
Prices shape: (65085, 8)
Trades shape: (23, 7)


timestamp,buyer,seller,product,currency,price,quantity
i64,str,str,str,str,f64,i64
90200,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",10011.0,10
122000,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",10012.0,10
155300,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",10012.0,10
177700,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",9998.0,10
185900,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",10000.0,10


## Step 2: Orderbook 散点图（核心可视化）

In [60]:
# fair price 计算参数（外层 fair + 内层 fair）
REG_MIN_VOLUME = 15
VOL_THRESHOLD = 20.0
MAX_STALE_MS = 3000
HALF_SPREAD_CONST = 10.0

# inner fair 参数（无 EMA，仅吸附）
INNER_PRIOR_OFFSET = -0.5
INNER_CONFLICT_TOL = 0.75
INNER_OFFSET_MIN = -2.0
INNER_OFFSET_MAX = 1.0

outer_wall_mid_df, inner_wall_mid_df, wall_mid_df = build_outer_inner_wall_mid(
    prices_long=prices_long,
    prices_wide=prices_wide,
    product=PRODUCT,
    day=DAY,
    reg_min_volume=REG_MIN_VOLUME,
    vol_threshold=VOL_THRESHOLD,
    max_stale_ms=MAX_STALE_MS,
    half_spread_const=HALF_SPREAD_CONST,
    inner_prior_offset=INNER_PRIOR_OFFSET,
    inner_conflict_tol=INNER_CONFLICT_TOL,
    inner_offset_min=INNER_OFFSET_MIN,
    inner_offset_max=INNER_OFFSET_MAX,
)

In [61]:
# Orderbook 散点图 + wall mid + trades
print('Step 2A: 窗口截取版（非 resample）')

# 节选一个时间片段，避免一次性绘制全量数据导致内存压力
plot_start_ts = 756500     # 例如: 350000
plot_end_ts = None        # 例如: 450000
default_window = 10000   # 当 start/end 未指定时，默认只画前 300000ms

base_slice = prices_long.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in base_slice.columns:
    base_slice = base_slice.filter(pl.col('day') == DAY)

if base_slice.height > 0:
    ts_min = int(base_slice['timestamp'].min())
    ts_max = int(base_slice['timestamp'].max())
    if plot_start_ts is None:
        plot_start_ts = ts_min
    if plot_end_ts is None:
        plot_end_ts = min(ts_max, plot_start_ts + default_window)

print(f'Plot window: [{plot_start_ts}, {plot_end_ts}]')

fig = plot_orderbook_scatter(
    prices_long, trades, product=PRODUCT, day=DAY, wall_mid_df=wall_mid_df,
    start_ts=plot_start_ts,
    end_ts=plot_end_ts,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    use_resampler=False,
    # use_resampler=True,
    # use_resampler_verbose=False,
    # show_l1_only=True,
    # l1_near_tol=4.0,
    # l1_far_tol=7.0,
    # show_l1_diagonal_guides=False,
    # show_outer_bands=False,
    # upper_band=7.0,
    # lower_band=-7.0,
    # show_inner_bands=False,
    # inner_band_low=-2.0,
    # inner_band_high=2.0,
    # show_l1_grid=False,
    # l1_grid_mode='all',
    # l1_grid_step=0.5,
    # show_l1_volume_labels=False,
    # l1_label_min_abs_vol=10,
    # show_l1_volume_heat=False,
    # l1_heat_sigma=1.0,
    # l1_heat_bin=0.5,
    # show_cross_fair=False,
    # show_trade_vs_fair=False,
    # show_l1_price_baseline=False,
    # show_price_baseline=False,
    # baseline_mode='median',
    # baseline_halfwidth=10.0,
    # show_l1_at_baseline=False,
    # baseline_atol=0.25,
    # show_asymmetry=False,
    # asymmetry_tol=0.5,
    # extra_hover_columns=['fair_source', 'fair_confidence'],
)
fig.show()

print('Step 2B: 全量 + resample 版（不按窗口截）')
fig_resample = plot_orderbook_scatter(
    prices_long, trades, product=PRODUCT, day=DAY, wall_mid_df=wall_mid_df,
    start_ts=None,
    end_ts=None,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    use_resampler=True,
    # use_resampler_verbose=False,
    # show_l1_only=True,
    # l1_near_tol=4.0,
    # l1_far_tol=7.0,
    # show_l1_diagonal_guides=False,
    # show_outer_bands=False,
    # upper_band=7.0,
    # lower_band=-7.0,
    # show_inner_bands=False,
    # inner_band_low=-2.0,
    # inner_band_high=2.0,
    # show_l1_grid=False,
    # l1_grid_mode='all',
    # l1_grid_step=0.5,
    # show_l1_volume_labels=False,
    # l1_label_min_abs_vol=10,
    # show_l1_volume_heat=False,
    # l1_heat_sigma=1.0,
    # l1_heat_bin=0.5,
    # show_cross_fair=False,
    # show_trade_vs_fair=False,
    # show_l1_price_baseline=False,
    # show_price_baseline=False,
    # baseline_mode='median',
    # baseline_halfwidth=10.0,
    # show_l1_at_baseline=False,
    # baseline_atol=0.25,
    # show_asymmetry=False,
    # asymmetry_tol=0.5,
    # extra_hover_columns=['fair_source', 'fair_confidence'],
)
fig_resample.show()

Step 2A: 窗口截取版（非 resample）
Plot window: [756500, 766500]


Step 2B: 全量 + resample 版（不按窗口截）


## Step 3: Normalized Orderbook（去趋势，Outer Fair vs Inner Fair）

In [62]:
print('Step 3A: Normalized by OUTER fair')
fig_outer = plot_normalized_orderbook(
    prices_long, outer_wall_mid_df, product=PRODUCT, day=DAY, trades=trades,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    overlay_wall_mid_trend=True,
)
fig_outer.update_layout(title=f'{PRODUCT} Normalized Orderbook (Outer Fair)')
fig_outer.show()

print('Step 3B: Normalized by INNER fair')
fig_inner = plot_normalized_orderbook(
    prices_long, inner_wall_mid_df, product=PRODUCT, day=DAY, trades=trades,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    overlay_wall_mid_trend=True,
)
fig_inner.update_layout(title=f'{PRODUCT} Normalized Orderbook (Inner Fair)')
fig_inner.show()

Step 3A: Normalized by OUTER fair


Step 3B: Normalized by INNER fair


## Step 4: Spread 统计

In [63]:
spread_df = compute_spread(prices_wide.filter(pl.col('product') == PRODUCT))
print('Wall spread stats:')
print(spread_df['wall_spread'].describe())
print(f"\nWall mid vs Raw mid diff: {(spread_df['wall_mid'] - spread_df['raw_mid']).mean():.4f}")

Wall spread stats:
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 9257.0    │
│ null_count ┆ 743.0     │
│ mean       ┆ 20.142703 │
│ std        ┆ 1.473861  │
│ min        ┆ 6.0       │
│ 25%        ┆ 19.0      │
│ 50%        ┆ 21.0      │
│ 75%        ┆ 21.0      │
│ max        ┆ 22.0      │
└────────────┴───────────┘

Wall mid vs Raw mid diff: 0.0447


## Step 5: Return Autocorrelation

In [64]:
wm = wall_mid_df.sort('timestamp')['wall_mid'].to_numpy()
acf, ci = return_autocorrelation(wm, max_lag=20)

fig = plot_autocorrelation(acf, ci, title=f'{PRODUCT} Return ACF')
fig.show()

# 解读
print(f'ACF(1) = {acf[0]:.4f}, 95% CI = +/-{ci:.4f}')
if abs(acf[0]) < ci:
    print('-> 不可预测，专注做市')
elif acf[0] < -ci:
    print('-> 短期均值回复，可以逆向交易')
else:
    print('-> 短期动量，taker 有 informed signal')

ACF(1) = -0.1292, 95% CI = +/-0.0196
-> 短期均值回复，可以逆向交易


## Step 6: 交易者分类

In [65]:
product_trades = trades.filter(pl.col('product') == PRODUCT)

if 'buyer' in trades.columns:
    fig = plot_trade_profile(trades, product=PRODUCT)
    fig.show()
    
    # 每个 trader 的行为模式
    for trader in product_trades['buyer'].unique().to_list():
        if not trader:
            continue
        tt = product_trades.filter(pl.col('buyer') == trader)
        print(f"\n{trader}: {len(tt)} buys, qty mode={tt['quantity'].mode().to_list()}, "
              f"avg_price={tt['price'].mean():.1f}")
else:
    print('No buyer/seller info in this round')

## Step 7: 订单间隔 / 成交间隔分布

In [66]:
order_intervals = order_interval_distribution(
    prices_long,
    product=PRODUCT,
    day=DAY,
    min_order_volume=MIN_ORDER_VOLUME if FILTER_BY_VOLUME else None,
)
trade_intervals = trade_interval_distribution(
    trades,
    product=PRODUCT,
    day=DAY,
    min_trade_quantity=MIN_TRADE_QUANTITY if FILTER_BY_VOLUME else None,
)

print(f"Order intervals count: {order_intervals.height}, mean={order_intervals['interval_ms'].mean() if order_intervals.height else 'NA'}")
print(f"Trade intervals count: {trade_intervals.height}, mean={trade_intervals['interval_ms'].mean() if trade_intervals.height else 'NA'}")

wm_overlay = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_overlay.columns:
    wm_overlay = wm_overlay.filter(pl.col('day') == DAY)

fig_o = plot_interval_distribution(
    order_intervals,
    title=f"Order Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_o.show()

fig_t = plot_interval_distribution(
    trade_intervals,
    title=f"Trade Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_t.show()

Order intervals count: 9981, mean=100.18034265103697
Trade intervals count: 22, mean=38904.545454545456


## Step 8: Wall Mid 奇偶统计

In [67]:
wm_stats = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_stats.columns:
    wm_stats = wm_stats.filter(pl.col('day') == DAY)
wm_stats = wm_stats.filter(pl.col('wall_mid').is_not_null()).with_columns(
    (2*pl.col('wall_mid') % 2 == 0).alias('is_even')
)

ratio_tbl = wm_stats.group_by('is_even').agg(pl.len().alias('n')).with_columns(
    (pl.col('n') / pl.col('n').sum()).alias('ratio')
).sort('is_even')

print('2 * Wall mid 奇偶占比:')
print(ratio_tbl)

trades_with_wm = (
    trades.filter(pl.col('product') == PRODUCT)
    .join(wm_stats.select(['timestamp', 'product', 'wall_mid', 'is_even']), on=['timestamp', 'product'], how='left')
    .filter(pl.col('is_even').is_not_null())
)

qty_col = 'quantity' if 'quantity' in trades_with_wm.columns else ('volume' if 'volume' in trades_with_wm.columns else None)
if qty_col is None:
    print('Trades 中没有 quantity/volume 列，无法统计成交量。')
else:
    trade_qty_tbl = (
        trades_with_wm.group_by('is_even')
        .agg([
            pl.col(qty_col).abs().sum().alias('total_trade_qty'),
            pl.len().alias('trade_count'),
        ])
        .sort('is_even')
    )
    print('\nWall mid 奇偶时的总成交量:')
    print(trade_qty_tbl)

even_ratio = ratio_tbl.filter(pl.col('is_even') == True)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == True).height else 0.0
odd_ratio = ratio_tbl.filter(pl.col('is_even') == False)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == False).height else 0.0
print(f"\nEven ratio={even_ratio:.4f}, Odd ratio={odd_ratio:.4f}, Δfrom 50%={abs(even_ratio-0.5):.4f}")

2 * Wall mid 奇偶占比:
shape: (2, 3)
┌─────────┬──────┬────────┐
│ is_even ┆ n    ┆ ratio  │
│ ---     ┆ ---  ┆ ---    │
│ bool    ┆ u32  ┆ f64    │
╞═════════╪══════╪════════╡
│ false   ┆ 9003 ┆ 0.9003 │
│ true    ┆ 997  ┆ 0.0997 │
└─────────┴──────┴────────┘

Wall mid 奇偶时的总成交量:
shape: (2, 3)
┌─────────┬─────────────────┬─────────────┐
│ is_even ┆ total_trade_qty ┆ trade_count │
│ ---     ┆ ---             ┆ ---         │
│ bool    ┆ i64             ┆ u32         │
╞═════════╪═════════════════╪═════════════╡
│ false   ┆ 210             ┆ 21          │
│ true    ┆ 20              ┆ 2           │
└─────────┴─────────────────┴─────────────┘

Even ratio=0.0997, Odd ratio=0.9003, Δfrom 50%=0.4003


## Step 9: 标准化档位成交统计

In [68]:
level_stats = normalized_level_trade_stats(
    trades=trades,
    wall_mid_df=wall_mid_df,
    product=PRODUCT,
    day=DAY,
    round_to=None,
)

print('标准化档位成交统计（norm_level = trade_price - wall_mid）:')
print(level_stats)

if level_stats.height:
    print('\n按成交次数排序 Top10:')
    print(level_stats.sort('trade_count', descending=True).head(10))

# 验证：norm_level 是否符合 0.5 网格，以及 .0/.5 占比
norm = level_stats.select(['norm_level', 'trade_count']).with_columns([
    (pl.col('norm_level') * 2).round(8).alias('norm_x2'),
]).with_columns([
    ((pl.col('norm_x2') - pl.col('norm_x2').round(0)).abs() < 1e-8).alias('is_half_grid'),
    (pl.col('norm_x2').round(0) % 2 == 0).alias('is_integer_level'),
])

total_trades = norm['trade_count'].sum() if norm.height else 0
on_grid_trades = norm.filter(pl.col('is_half_grid'))['trade_count'].sum() if norm.height else 0
int_trades = norm.filter(pl.col('is_half_grid') & pl.col('is_integer_level'))['trade_count'].sum() if norm.height else 0
half_trades = norm.filter(pl.col('is_half_grid') & (~pl.col('is_integer_level')))['trade_count'].sum() if norm.height else 0

print(f"\nHalf-grid integrity: {on_grid_trades}/{total_trades} = {on_grid_trades/total_trades if total_trades else 0:.4f}")
print(f"Integer-level trades (.0): {int_trades} ({int_trades/total_trades if total_trades else 0:.4f})")
print(f"Half-level trades (.5): {half_trades} ({half_trades/total_trades if total_trades else 0:.4f})")

标准化档位成交统计（norm_level = trade_price - wall_mid）:
shape: (9, 4)
┌────────────┬─────────────┬────────────────┬──────────────┐
│ norm_level ┆ trade_count ┆ total_quantity ┆ avg_quantity │
│ ---        ┆ ---         ┆ ---            ┆ ---          │
│ f64        ┆ u32         ┆ i64            ┆ f64          │
╞════════════╪═════════════╪════════════════╪══════════════╡
│ -10.5      ┆ 3           ┆ 30             ┆ 10.0         │
│ -8.5       ┆ 2           ┆ 20             ┆ 10.0         │
│ -8.0       ┆ 1           ┆ 10             ┆ 10.0         │
│ -7.5       ┆ 6           ┆ 60             ┆ 10.0         │
│ 7.5        ┆ 4           ┆ 40             ┆ 10.0         │
│ 8.0        ┆ 1           ┆ 10             ┆ 10.0         │
│ 8.5        ┆ 4           ┆ 40             ┆ 10.0         │
│ 9.5        ┆ 1           ┆ 10             ┆ 10.0         │
│ 10.5       ┆ 1           ┆ 10             ┆ 10.0         │
└────────────┴─────────────┴────────────────┴──────────────┘

按成交次数排序 Top10:
shape: 

## Step 10: Adverse Selection 检验

In [69]:
wm_series = wall_mid_df.filter(pl.col('product') == PRODUCT).select(['timestamp', 'wall_mid'])
as_result = adverse_selection(product_trades, wm_series)

print('Adverse Selection (taker PnL after trade):')
for h, v in as_result.items():
    label = 'SAFE (taker=noise)' if v <= 0 else 'DANGER (taker=informed)'
    print(f'  h={h:3d} steps: {v:+.4f}  {label}')

Adverse Selection (taker PnL after trade):
  h=  1 steps: +0.1304  DANGER (taker=informed)
  h=  5 steps: -0.1087  SAFE (taker=noise)
  h= 10 steps: +0.0652  DANGER (taker=informed)
  h= 50 steps: -0.8043  SAFE (taker=noise)


## 结论

在这里写你的观察...

In [70]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox


def _pick_inner_fair_series():
    candidates = []

    if 'fair_dual_df' in globals():
        try:
            pdf = fair_dual_df.to_pandas() if hasattr(fair_dual_df, 'to_pandas') else fair_dual_df
            candidates.append(('fair_dual_df', pdf))
        except Exception as e:
            print('fair_dual_df not usable:', e)

    if 'inner_wall_mid_df' in globals():
        try:
            pdf = inner_wall_mid_df.to_pandas() if hasattr(inner_wall_mid_df, 'to_pandas') else inner_wall_mid_df
            candidates.append(('inner_wall_mid_df', pdf))
        except Exception as e:
            print('inner_wall_mid_df not usable:', e)

    for name, df in candidates:
        cols = [c for c in df.columns if 'inner' in str(c).lower() and ('fair' in str(c).lower() or 'mid' in str(c).lower())]
        if cols:
            col = cols[0]
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(s) > 50:
                return name, col, s

    raise ValueError('No suitable inner fair series found in current kernel. Please run upstream cells first.')

src_name, col_name, px = _pick_inner_fair_series()

# Use first difference as return proxy for price-level fair values.
ret = px.diff().dropna()
abs_ret = ret.abs()
sq_ret = ret.pow(2)

print(f'Using series: {src_name}.{col_name}')
print(f'n_obs(px)={len(px)}, n_obs(ret)={len(ret)}')
print(f'zero-return ratio={(ret == 0).mean():.4f}')

lags = [1, 5, 10, 20]
acf_abs = acf(abs_ret, nlags=max(lags), fft=True)
acf_sq = acf(sq_ret, nlags=max(lags), fft=True)

print('\nACF(abs returns):')
for L in lags:
    print(f'lag {L:>2}: {acf_abs[L]: .4f}')

print('\nACF(squared returns):')
for L in lags:
    print(f'lag {L:>2}: {acf_sq[L]: .4f}')

lb_abs = acorr_ljungbox(abs_ret, lags=lags, return_df=True)
lb_sq = acorr_ljungbox(sq_ret, lags=lags, return_df=True)

print('\nLjung-Box on abs returns:')
print(lb_abs)

print('\nLjung-Box on squared returns:')
print(lb_sq)

sig_abs = (lb_abs['lb_pvalue'] < 0.05).sum()
sig_sq = (lb_sq['lb_pvalue'] < 0.05).sum()

print('\nConclusion (5% level):')
if sig_abs >= 2 or sig_sq >= 2:
    print('Evidence supports volatility clustering in inner fair.')
else:
    print('No strong evidence of volatility clustering in inner fair.')

ValueError: No suitable inner fair series found in current kernel. Please run upstream cells first.

In [ ]:
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

ret_nz = ret[ret != 0]
abs_ret_nz = ret_nz.abs()
sq_ret_nz = ret_nz.pow(2)

print(f'n_obs(non-zero ret)={len(ret_nz)}')

lags = [1, 5, 10, 20]
lb_abs_nz = acorr_ljungbox(abs_ret_nz, lags=lags, return_df=True)
lb_sq_nz = acorr_ljungbox(sq_ret_nz, lags=lags, return_df=True)

print('\nLjung-Box on abs returns (non-zero only):')
print(lb_abs_nz)

print('\nLjung-Box on squared returns (non-zero only):')
print(lb_sq_nz)

# ARCH LM test on original return series
arch_lm = het_arch(ret.dropna(), nlags=10)
print('\nARCH-LM (ret, 10 lags):')
print(f'LM stat={arch_lm[0]:.4f}, LM pvalue={arch_lm[1]:.6f}, F stat={arch_lm[2]:.4f}, F pvalue={arch_lm[3]:.6f}')

n_obs(non-zero ret)=2757

Ljung-Box on abs returns (non-zero only):
       lb_stat     lb_pvalue
1   283.616667  1.223149e-63
5   298.090320  2.577588e-62
10  307.233518  4.592130e-60
20  311.261251  4.029599e-54

Ljung-Box on squared returns (non-zero only):
       lb_stat     lb_pvalue
1   219.071426  1.441887e-49
5   229.207171  1.581715e-47
10  238.753039  1.251913e-45
20  244.548847  1.432370e-40

ARCH-LM (ret, 10 lags):
LM stat=18.4479, LM pvalue=0.047862, F stat=1.8462, F pvalue=0.047801


In [ ]:
# Step X: fair 上下方 × 理性/非理性 的订单次数/订单量到达率（8项）
# 统一只用 tick 口径：每个 timestamp 视为 1 tick。

fair_src = wall_mid_df.select(['timestamp', 'product', 'wall_mid'])

orders = prices_long.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in orders.columns:
    orders = orders.filter(pl.col('day') == DAY)

if FILTER_BY_VOLUME:
    orders = orders.filter(pl.col('volume').abs() >= MIN_ORDER_VOLUME)

side_expr = (
    pl.col('side')
    if 'side' in orders.columns
    else pl.when(pl.col('volume') > 0).then(pl.lit('bid')).otherwise(pl.lit('ask'))
)

orders = (
    orders
    .join(
        fair_src.filter(pl.col('product') == PRODUCT).select(['timestamp', 'wall_mid']),
        on='timestamp',
        how='left',
    )
    .filter(pl.col('wall_mid').is_not_null())
    .with_columns([
        (pl.col('price') - pl.col('wall_mid')).alias('delta'),
        side_expr.alias('side_used'),
    ])
    .filter(pl.col('delta') != 0)
    .with_columns([
        pl.when(pl.col('delta') > 0).then(pl.lit('above')).otherwise(pl.lit('below')).alias('position'),
        (
            ((pl.col('side_used') == 'bid') & (pl.col('delta') < 0)) |
            ((pl.col('side_used') == 'ask') & (pl.col('delta') > 0))
        ).alias('is_rational')
    ])
    .with_columns([
        pl.when(pl.col('is_rational')).then(pl.lit('rational')).otherwise(pl.lit('irrational')).alias('rationality'),
        pl.col('volume').abs().alias('abs_volume'),
    ])
)

ts_count = orders.select(pl.col('timestamp').n_unique().alias('n_ts')).item()
if ts_count == 0:
    print('没有可用订单（可能过滤条件过严或 fair 对齐后为空）。')
else:
    grouped = (
        orders
        .group_by(['position', 'rationality'])
        .agg([
            pl.len().alias('order_count'),
            pl.col('abs_volume').sum().alias('total_volume'),
        ])
        .with_columns([
            (pl.col('order_count') / ts_count).alias('count_rate_per_tick'),
            (pl.col('total_volume') / ts_count).alias('volume_rate_per_tick'),
        ])
    )

    keys = [
        ('above', 'rational'),
        ('above', 'irrational'),
        ('below', 'rational'),
        ('below', 'irrational'),
    ]

    print(f'PRODUCT={PRODUCT}, DAY={DAY}, tick样本数={ts_count}')
    print('--- 8项结果（position × rationality × tick口径）---')

    for pos, rat in keys:
        sub = grouped.filter((pl.col('position') == pos) & (pl.col('rationality') == rat))
        if sub.height == 0:
            c_tick = 0.0
            v_tick = 0.0
        else:
            c_tick = float(sub['count_rate_per_tick'][0])
            v_tick = float(sub['volume_rate_per_tick'][0])

        print(f'{pos:>5} | {rat:>10} | count/tick={c_tick:.6f}')
        print(f'{pos:>5} | {rat:>10} | volume/tick={v_tick:.6f}')

PRODUCT=ASH_COATED_OSMIUM, DAY=0, tick样本数=9982
--- 8项结果（position × rationality × tick口径）---
above |   rational | count/tick=1.609197
above |   rational | volume/tick=30.237828
above | irrational | count/tick=0.001503
above | irrational | volume/tick=0.015027
below |   rational | count/tick=1.595472
below |   rational | volume/tick=30.008816
below | irrational | count/tick=0.002104
below | irrational | volume/tick=0.021038
